# Khmer word detection v2 (fine-tune)
Continues from v1 (`~/khmer-models/khmer_word_det/weights/best.pt`) and trains on **old synthetic data + your real data**.
Put real data on the school machine as `data/real/images/*.png` + `data/real/labels/*.txt` (YOLO format, class 0), e.g. via the Jupyter upload button.

In [ ]:
import os, subprocess
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)

In [ ]:
%pip install -q ultralytics kagglehub

In [ ]:
import torch; assert torch.cuda.is_available(), 'no CUDA'; print(torch.cuda.get_device_name(0))

## Data: old (same split as v1) + real

In [ ]:
import os, random, shutil, yaml
from pathlib import Path
import kagglehub

def put(a, b):
    b.parent.mkdir(parents=True, exist_ok=True)
    if not b.exists():
        try: os.link(a, b)
        except OSError: shutil.copy2(a, b)

def split(img_dir, lab_dir, out, seed=0):
    stems = sorted(p.stem for p in Path(img_dir).glob('*.png') if (Path(lab_dir)/f'{p.stem}.txt').exists())
    random.Random(seed).shuffle(stems)
    n = len(stems)
    cut = {'train': stems[:int(.8*n)], 'val': stems[int(.8*n):int(.9*n)], 'test': stems[int(.9*n):]}
    for k, ss in cut.items():
        for s in ss:
            put(Path(img_dir)/f'{s}.png', out/'images'/k/f'{s}.png')
            put(Path(lab_dir)/f'{s}.txt', out/'labels'/k/f'{s}.txt')
    return {k: len(v) for k, v in cut.items()}

src = Path(kagglehub.dataset_download('emhengly/khmer-text-detection-dataset'))
OLD = Path('khmer-det').resolve()      # same seed/split as v1, so the old test set stays unseen
print('old :', split(src/'PNG_Files'/'PNG_Files', src/'labels_yolov8_format'/'labels_yolov8_format', OLD))
REAL = Path('khmer-real-split').resolve()
print('real:', split('data/real/images', 'data/real/labels', REAL))

def yml(path, **kw):
    Path(path).write_text(yaml.safe_dump({'names': {0: 'khmer_word'}, **kw}, sort_keys=False)); return str(Path(path).resolve())
DATA_MIX  = yml('data_v2.yaml',   train=[str(OLD/'images/train'), str(REAL/'images/train')], val=[str(OLD/'images/val'), str(REAL/'images/val')])
DATA_OLD  = yml('data_old_test.yaml',  train=str(OLD/'images/train'), val=str(OLD/'images/test'))
DATA_REAL = yml('data_real_test.yaml', train=str(REAL/'images/train'), val=str(REAL/'images/test'))

## Train (fine-tune from v1 weights)
Lower learning rate than v1. Raise `EPOCHS` if real-data mAP is still rising.

In [ ]:
from ultralytics import YOLO
V1 = Path.home()/'khmer-models/khmer_word_det/weights/best.pt'
assert V1.exists(), f'missing {V1}'
EPOCHS, IMGSZ, BATCH = 30, 640, 24
OUT_DIR = Path.home()/'khmer-models'
RUN = OUT_DIR/'khmer_word_det_v2'
BEST, LAST = RUN/'weights/best.pt', RUN/'weights/last.pt'

if LAST.exists():
    try: YOLO(LAST).train(resume=True)
    except Exception as e: print('nothing to resume:', e)
else:
    YOLO(V1).train(data=DATA_MIX, epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=0,
                   project=str(OUT_DIR), name='khmer_word_det_v2', exist_ok=True,
                   optimizer='SGD', lr0=0.002, warmup_epochs=1, patience=10, save_period=5)
print(BEST, BEST.exists())

## Validate: old test vs real test (v1 vs v2)

In [ ]:
def score(w, data):
    m = YOLO(w).val(data=data, split='val', imgsz=IMGSZ, iou=0.5, verbose=False)
    return f'P={m.box.mp:.3f} R={m.box.mr:.3f} mAP50={m.box.map50:.3f} mAP50-95={m.box.map:.3f}'
for name, w in [('v1', V1), ('v2', BEST)]:
    print(name, 'old test :', score(w, DATA_OLD))
    print(name, 'real test:', score(w, DATA_REAL))

In [ ]:
from IPython.display import Image, display
for f in ['results.png', 'BoxPR_curve.png', 'BoxF1_curve.png', 'confusion_matrix.png']:
    display(Image(str(RUN/f), width=800))

## Predict on a few real test pages

In [ ]:
best = YOLO(BEST)
imgs = sorted((REAL/'images'/'test').glob('*.png'))[:3]
for r in best.predict(imgs, imgsz=IMGSZ, conf=0.25, save=True, project='runs', name='predict_v2', exist_ok=True):
    display(Image(str(Path(r.save_dir)/Path(r.path).name), width=700))

## Export weights
On the laptop: `python scripts/pull_weights.py` (edit its folder to `khmer_word_det_v2` for these weights).

In [ ]:
print(BEST, LAST)